# Практика: кластеры и список аномалий

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def find_orders_csv():
    for path in (
        Path("orders_slim.csv"),
        Path("../orders_slim.csv"),
        Path("../../data/orders_slim.csv"),
        Path("../data/orders_slim.csv"),
        Path("../../../data/orders_slim.csv"),
    ):
        if path.exists():
            return path.resolve()
    return (
        "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/"
        "modules/08_08_logistics_clustering/data/orders_slim.csv"
    )


CSV_PATH = find_orders_csv()
DATE_COLUMNS = [
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
    "order_delivered_customer_date",
]
df = pd.read_csv(CSV_PATH, parse_dates=DATE_COLUMNS)
assert len(df) > 0
assert df["order_id"].notna().all()
print(f"Загружено заказов: {len(df)}")

from sklearn.cluster import DBSCAN, KMeans
from sklearn.preprocessing import StandardScaler

FEATURES = ["delivery_days", "freight_value", "delay_days"]


## 1. Подготовка без утечки смысла

Масштабируйте только три FEATURES; is_late не входит.

In [ ]:
X = None  # TODO
Xs = None  # TODO
assert list(X.columns) == FEATURES
assert "is_late" not in X.columns and Xs.shape == X.shape


## 2. KMeans-сегменты

In [ ]:
labels_km = None  # TODO: k=3, seed=54, n_init=10
assert len(labels_km) == len(df)
assert len(set(labels_km)) == min(3, len(df))


## 3. DBSCAN-шум

In [ ]:
labels_db = None  # TODO: eps=0.9, min_samples=4
assert len(labels_db) == len(df)
assert set(np.unique(labels_db))


## 4. Общая таблица

In [ ]:
clustered = None  # TODO: копия df + cluster_km + cluster_db
assert isinstance(clustered, pd.DataFrame) and len(clustered) == len(df)
assert {"cluster_km", "cluster_db"} <= set(clustered.columns)


## 5. Профили для ops

Средние, медианы, размер и late-rate для описания.

In [ ]:
profiles = None  # TODO
assert isinstance(profiles, pd.DataFrame)
assert {"size", "delivery_days_mean", "freight_median", "delay_mean", "late_rate"} <= set(profiles.columns)


## 6. Кандидаты DBSCAN

Метка -1 — кандидат, но не автоматическая ошибка.

In [ ]:
noise_candidates = None  # TODO
assert isinstance(noise_candidates, pd.DataFrame)
assert set(noise_candidates.index) <= set(clustered.index)
assert (noise_candidates["cluster_db"] == -1).all()


## 7. Страховка для малого набора

Добавьте top-5 delay к DBSCAN-кандидатам.

In [ ]:
top_delay_idx = None  # TODO
candidate_idx = None  # TODO: объединение set
assert len(top_delay_idx) == min(5, len(df))
assert set(top_delay_idx) <= candidate_idx


## 8. Ранжирование аномалий

Создайте severity из стандартизированных delay/freight.

In [ ]:
anomalies = None  # TODO: кандидаты, severity, сортировка
assert isinstance(anomalies, pd.DataFrame) and len(anomalies) >= min(5, len(df))
assert "severity" in anomalies.columns
assert anomalies["severity"].is_monotonic_decreasing


## 9. Обоснование каждой строки

Колонка reason с конкретным наблюдением.

In [ ]:
reasons = None  # TODO: список строк той же длины
assert len(reasons) == len(anomalies)
assert all(isinstance(text, str) and len(text) >= 35 for text in reasons)
anomalies = anomalies.assign(reason=reasons)


## 10. Самостоятельно: записка хабу

In [ ]:
OPS_NOTE = ""  # TODO: 250+ символов, число сегментов/кандидатов, действие, ограничение
assert len(OPS_NOTE) >= 250
assert str(len(anomalies)) in OPS_NOTE
assert "is_late" in OPS_NOTE
